In [2]:
%pip install pandas
%pip install scikit-learn
%pip install xgboost
%pip install tensorflow
%pip install matplotlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from collections import defaultdict
from xgboost import XGBRegressor
import tensorflow
from tensorflow.keras import models, layers
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt 
from tensorflow.keras.optimizers import Adam
from pathlib import Path

from sklearn.metrics import mean_squared_error
import numpy as np
import math

import json
import os

Supplementary Data Processing

In [4]:
def processed_holidays_events():
    """
    Returns an embedded dictionary with has_holiday[data][locale] returning True or False

    Returns
    -------
    has_holiday: dict
        The dictionary returning whether theres a holiday on that given day
    """

    holidays_events = pd.read_csv("data/holidays_events.csv")
    holidays_events = holidays_events[holidays_events["transferred"] == False]
    holidays_events = holidays_events.drop(columns=["description"])

    date_grouped_holidays = holidays_events.groupby("date")

    has_holiday = defaultdict(lambda: defaultdict(lambda: False)) #Defaults to {date: {city: False}}
    for date, grouped_table in date_grouped_holidays:
        if "National" in grouped_table["locale"].unique():
            has_holiday[date] = defaultdict(lambda: True)
            continue
        for city in grouped_table["locale_name"].to_list():
            has_holiday[date][city] = True

    return has_holiday

In [5]:
def process_oil_prices():
    """
    Returns a dictionary with date pointing to oil prices.

    Returns
    -------
    oil_prices: dict
        The dictionary with schema {date : oil_price}
    """
    
    oil_prices = pd.read_csv("data/oil.csv").set_index("date")
    return oil_prices.to_dict()['dcoilwtico']

In [6]:
def process_stores_table():
    """
    Processes and returns the stores table

    Returns
    -------
    stores: pd.DataFrame 
    """
    
    stores = pd.read_csv("data/stores.csv")

    stores["type"] = stores.apply(
        lambda row: "Other" if row["type"] not in ["D", "C"] else row["type"], axis=1
    )
    
    stores["cluster"] = stores["cluster"].astype("category")
    stores["type"] = stores["type"].astype("category")

    return stores


In [7]:
def process_transactions():
    """
    Returns a dictionary with date and store_nbr pointing to transactions

    Returns
    -------
    key_to_transactions: dict[str:int]
    """

    transactions_df = pd.read_csv("data/transactions.csv")
    transactions_df["store_nbr"] = transactions_df["store_nbr"].astype("string")
    transactions_df["temp"] = transactions_df["date"] + transactions_df["store_nbr"] 

    key_to_transactions = defaultdict(int)

    for _, row in transactions_df.iterrows():
        key_to_transactions[row["temp"]] = row["transactions"]
    
    return key_to_transactions

Time Series Data Processing

In [50]:
def process_lstm_data(data, grouping_category = 1):
    """
    Processes store price data and groups by store_nbr, family

    Paramters
    ---------
    data: pd.DataFrame
        The dataframe containing the time series data
    
    Returns
    -------
    key_to_df : dict[str: pd.DataFrame]
        dictionary with (store_nbr, family) pointing to the grouped dataframe
    """
    
    if grouping_category == 1:
        grouped_data =  data.groupby(["store_nbr", "family"], observed=False)
    elif grouping_category == 2:
        grouped_data = data.groupby("store_nbr", observed=False)
    elif grouping_category == 3:
        grouped_data = data.groupby("family", observed=False)
        
    key_to_df = {}
    key_to_scale_factors = {}
    for key, grouped_df in grouped_data:
        grouped_df = grouped_df[["date", "sales", "onpromotion", "has_holiday", "oil_price"]].copy()

        if grouping_category != 1:

            grouped_df = (
                grouped_df.groupby("date")
                        .agg({
                            "sales": "mean",
                            "onpromotion": "mean",
                            "has_holiday": "first",
                            "oil_price": "first",
                        })
            )

            grouped_df = grouped_df.rename_axis("date_index")
            grouped_df["date"] = grouped_df.index
        
        scale_factor = max(grouped_df["sales"].max(), 1)
        grouped_df["sales"] = grouped_df["sales"] / max(grouped_df["sales"].max(), 1)

        key_to_scale_factors[key] = scale_factor
        key_to_df[key] = grouped_df

    return key_to_df, key_to_scale_factors
    

In [9]:
def create_sequences(data_array, seq_len):
    """ 
    Sequentializes an array and returns the sequnces

    Paramters
    ---------
    data_array: np.array
        The array to sequentialize
    seq_len: int
        The length of each sequnce
    
    Returns 
    -------
    X_seq: np.array
        A 2D array containing the sequences from time t-seq_len to time t-1
    y_seq: np.array
        A 1D array containing the value at time t
    """

    X_seq, y_seq = [], []

    for i in range(len(data_array) - seq_len):
        X_seq.append(data_array[i:i+seq_len])
        y_seq.append(data_array[i+seq_len, 0])
    
    return np.array(X_seq), np.array(y_seq).reshape(-1,1)

In [38]:
def preprocess_lstm_data(grouped_data, seq_len, additional_features = []):
    """Preprocesses data to be in LSTM  compatible format and engineers features"""
    
    # Creating features
    grouped_data["roc"] = grouped_data["sales"].pct_change().fillna(0).replace([np.inf, -np.inf], 0)
    grouped_data['rolling_avg'] = grouped_data['sales'].rolling(window=5, center=False).mean().fillna(0)
    grouped_data['rolling_std'] = grouped_data['sales'].rolling(window=5).std().fillna(0)
    grouped_data["has_holiday"] = grouped_data["has_holiday"].astype(int)

    # Create Sequence values
    grouped_data = grouped_data.sort_values("date")
    features = ["sales", "roc", "rolling_avg", "rolling_std", "has_holiday", "onpromotion", "oil_price"]
    features += additional_features

    data_array = grouped_data[features].to_numpy()

    X, y = create_sequences(data_array, seq_len)

    return X, y

In [11]:
def train_lstm_model(grouped_data, seq_len):
    """ 
    Initializes a 1D LSTM model trained on grouped (store_nbr, family) data

    Paramters
    ---------
    grouped_data: pd.DataFrame
        The time series data to train on
    seq_len: int
        The window length to train the model on
    
    Returns
    -------
    model: models.Sequential
        The trained LSTM Model
    val_rmse: float
        The Validation Root Mean Squared Error of this model
    index_to_lstm_predictions: dict[int:float]
        The dictionary holding the LSTM prediction value
    """
    
    X, y = preprocess_lstm_data(grouped_data, seq_len)

    # Simple train / test split
    split = int(0.75 * len(X))
    X_train, X_val = X[:split], X[split:]
    y_train, y_val = y[:split], y[split:]

    #Adapting the noramalization layer
    normalizer = layers.Normalization(axis=-1)
    normalizer.adapt(X_train)

    # Initialize the model
    num_features = 7
    hidden_dim = 64

    model = models.Sequential([ 
        layers.Input(shape=(seq_len, num_features)), 
        normalizer,
        layers.LSTM(hidden_dim, dropout=0.2), 
        layers.Dense(1, activation='linear')
    ])

    # Compile the model
    model.compile(
        optimizer=Adam(learning_rate=0.01), 
        loss='mse', 
        metrics=[tensorflow.keras.metrics.RootMeanSquaredError()]
    )   

    # Train the model
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=50,
        batch_size=64,
        verbose=0
    )

    # Validate the model
    train_rmse = history.history['root_mean_squared_error'][-1]
    val_rmse   = history.history['val_root_mean_squared_error'][-1]

    return model, val_rmse, train_rmse


In [12]:
def populate_lstm_predictions(grouped_data, key):
    """
    Runs inference on time series data and populates the X_train dataframe using indices

    Paramters
    ---------
    key: string
        The grouping of the data to populate
    
    Returns
    -------
    index_to_pred: dict[int:float]
        The dictionary representing the model predictions
    """

    try:
        lstm_model = tensorflow.keras.models.load_model(f"my_models/specialist_model_{key}")
    except OSError as e:
        print(f"Model for {key} does not exist")
        return {}
    
    seq_len = lstm_model.input_shape[1]
    num_features = lstm_model.input_shape[2]
    
    X, y = preprocess_lstm_data(grouped_data, seq_len)

    y_pred = lstm_model.predict(X, verbose=0)

    predictions = [np.nan]*seq_len + y_pred.flatten().tolist()

    populated_df = grouped_data.copy()
    populated_df["lstm_forecast"] = predictions
    
    return populated_df["lstm_forecast"].to_dict()

In [13]:
def plot_lstm_predictions(data):
    """Plots the LSTM Forecast for that specific (store_nbr, family) pair."""

    dates = data["date"]
    y_true = data["sales"]
    y_pred = data["lstm_forecast"]

    plt.figure(figsize=(12,6))
    plt.plot(dates, y_true, label='Actual', marker='o')
    plt.plot(dates, y_pred, label='Predicted', marker='x')
    plt.xlabel('Date')
    plt.ylabel('Sales')
    plt.title('LSTM Predictions vs Actual Sales')
    plt.legend()
    plt.grid(True)
    plt.show()

In [14]:
def train_specialists(train_df, grouping_category=1, override=False):
    """Specialist LSTM Model training loop"""

    key_to_df = process_lstm_data(train_df, grouping_category)
    keys = list(key_to_df.keys())

    for key in keys:

        # Check if model already exists
        model_path = Path(f"my_models/specialist_model_{key}")
        if model_path.exists() and not override:
            continue 
        
        curr_df = key_to_df[key]
        del key_to_df[key]
        model, val_rmse, train_rmse = train_lstm_model(curr_df, 20)
        model.save(f"my_models/specialist_model_{key}")

        print(val_rmse, train_rmse)

In [15]:
def process_train_data(data, has_holiday_dict, oil_prices_dict, stores_df, key_to_transactions):
    """Process and returns train_df"""

    def has_holiday(date_city_region, has_holiday_dict):
        """Helper function to be used in preprocessing"""
        
        date, city, region = date_city_region.split("_")
        return has_holiday_dict[date][city] or has_holiday_dict[date][region]
    
    data = pd.merge(data, stores_df, how="left", on="store_nbr")

    data["date_city_region"] = data["date"] + "_" + data["city"] + "_" + data["state"]
    data["has_holiday"] = data["date_city_region"].apply(
        lambda x: has_holiday(x, has_holiday_dict)
    )

    data['weekday'] = pd.to_datetime(data['date']).dt.day_name().astype("category")

    data["oil_price"] = data["date"].map(oil_prices_dict)
    mean_oil_price = data["oil_price"].mean()
    data["oil_price"] = data["oil_price"].fillna(mean_oil_price)
    
    data["family"] = data["family"].astype("category")
    data["store_nbr"] = data["store_nbr"].astype("category")
    data["city"] = data["city"].astype("category")
    data["state"] = data["state"].astype("category")
    data["onpromotion"] = data["onpromotion"].astype("float")
    
    data = data.drop(columns=["date_city_region"])
    
    data["store_nbr"] = data["store_nbr"].astype("string")
    data["temp"] = data["date"] + data["store_nbr"]
    data["transactions"] = data["temp"].map(key_to_transactions)

    data["store_nbr"] = data["store_nbr"].astype(int)
    data = data.drop(columns=["temp"])

    return data

Specialist LSTM Training

In [111]:
# Fetching train data

train_df = pd.read_csv("data/train.csv")
train_df = train_df[train_df["sales"].notna()]

In [112]:
# Fetching supplamentary data

has_holiday_dict = processed_holidays_events()
oil_prices_dict = process_oil_prices()
stores_df = process_stores_table()
key_to_transactions = process_transactions()

In [113]:
train_df = process_train_data(train_df, has_holiday_dict, oil_prices_dict, stores_df, key_to_transactions)

In [ ]:
# Fetching scale factors by grouping category

_, scale_factors1 = process_lstm_data(train_df, grouping_category=1)
_, scale_factors2 = process_lstm_data(train_df, grouping_category=2)
_, scale_factors3 = process_lstm_data(train_df, grouping_category=3)

In [57]:
#train_specialists(train_df, grouping_category=1, override=True)

In [115]:
train_df["temp"] = train_df.apply(
    lambda x: (x["store_nbr"], x["family"]), axis=1
) 

index_to_key = train_df["temp"].to_dict()
train_df = train_df.drop(columns=["temp"])

def scale_df1(index_to_lstm_prediction):
    for index in index_to_lstm_prediction.keys():
        key = index_to_key[int(index)]
        scale_factor = scale_factors1[key]
        index_to_lstm_prediction[index] *= scale_factor
    
    return index_to_lstm_prediction

In [116]:
def populate_category1(override=False, scale=True):
    """Populates train_df with (store_nbr, family) model predictions"""

    index_to_lstm_prediction = {}
    key_to_store_family_df, _ = process_lstm_data(train_df, grouping_category = 1)
    keys1 = list(key_to_store_family_df.keys())

    for key in keys1:
        grouped_df = key_to_store_family_df[key]
        new_lstm_predictions = populate_lstm_predictions(grouped_df, key)
        index_to_lstm_prediction.update(new_lstm_predictions)
    
    if scale:
        index_to_lstm_prediction = scale_df1(index_to_lstm_prediction)

    # Upload predictions to JSON
    if override:
        subfolder = "predictions"
        file_name = "category_1.json" if not scale else "category_1_scaled.json"
        file_path = os.path.join(subfolder, file_name)

        with open(file_path, "w") as f:
            json.dump(index_to_lstm_prediction, f, indent=4)

#populate_category1(override=True)

In [117]:
# Loads the saved JSON and updates train_df

file_path = "predictions/category_1_scaled.json"

with open(file_path, "r") as f:
    index_to_lstm_prediction = json.load(f)

train_df["id"] = train_df["id"].astype(str)
train_df["store_family_prediction"] = train_df["id"].map(index_to_lstm_prediction)

In [ ]:
def scale_df2(store_to_date_prediction_dict):
    for store in store_to_date_prediction_dict.keys():
        scale_factor = scale_factors2[int(store)]
        for date in store_to_date_prediction_dict[store].keys():
            if math.isnan(store_to_date_prediction_dict[store][date]):
                continue
            store_to_date_prediction_dict[store][date] = float(store_to_date_prediction_dict[store][date])
            store_to_date_prediction_dict[store][date] *= scale_factor
    return store_to_date_prediction_dict

def populate_category2(override=False, scale=True):
    """Populates train_df with store_nbr model predictions"""

    store_to_date_prediction_dict = {} # 2D dictionary {store_nbr: {date: prediction}}
    key_to_store_df, _ = process_lstm_data(train_df, grouping_category = 2)
    keys2 = list(key_to_store_df.keys())

    for key in keys2:
        grouped_df = key_to_store_df[key]
        grouped_df = grouped_df.set_index("date", drop=False)
        grouped_df.index.name = "date_index"

        date_to_store_pred = populate_lstm_predictions(grouped_df, key)
        store_to_date_prediction_dict[key] = date_to_store_pred

    if scale:
        store_to_date_prediction_dict = scale_df2(store_to_date_prediction_dict)
    
    # Uploads the predictions to JSON
    if override:
        subfolder = "predictions"
        file_name = "category_2.json" if not scale else "category_2_scaled.json"
        file_path = os.path.join(subfolder, file_name)

        with open(file_path, "w") as f:
            json.dump(store_to_date_prediction_dict, f, indent=4)

#populate_category2(override=True)

In [119]:
# Loads the saved JSON and upadtes train_df

file_path = "predictions/category_2_scaled.json"

with open(file_path, "r") as f:
    store_to_date_prediction_dict = json.load(f)

train_df["store_prediction"] = train_df.apply(
    lambda x: store_to_date_prediction_dict[str(x["store_nbr"])][x["date"]], axis=1
)

In [ ]:
def scale_df3(family_to_date_prediction_dict):
    for family in family_to_date_prediction_dict.keys():
        scale_factor = scale_factors3[family]
        for date in family_to_date_prediction_dict[family].keys():
            if math.isnan(family_to_date_prediction_dict[family][date]):
                continue
            family_to_date_prediction_dict[family][date] = float(family_to_date_prediction_dict[family][date])
            family_to_date_prediction_dict[family][date] *= scale_factor
    return family_to_date_prediction_dict

def populate_category3(override=False, scale=True):
    """Populates train_df with family model predictions"""
    family_to_date_prediction_dict = {} # 2D dictionary {family: {date: prediction}}
    key_to_family_df, _ = process_lstm_data(train_df, grouping_category = 3)
    keys3 = list(key_to_family_df.keys())

    for key in keys3:

        grouped_df = key_to_family_df[key]
        grouped_df = grouped_df.set_index("date", drop=False)
        grouped_df.index.name = "date_index"
        
        date_to_pred = populate_lstm_predictions(grouped_df, key)
        family_to_date_prediction_dict[key] = date_to_pred
    
    if scale:
        family_to_date_prediction_dict = scale_df3(family_to_date_prediction_dict)

    if override:
        subfolder = "predictions"
        file_name = "category_3.json" if not scale else "category_3_scaled.json"

        file_path = os.path.join(subfolder, file_name)

        with open(file_path, "w") as f:
            json.dump(family_to_date_prediction_dict, f, indent=4)

#populate_category3(override=True)

In [121]:
# Loads the saved JSON and updates train_df

file_path = "predictions/category_3_scaled.json"

with open(file_path, "r") as f:
    family_to_date_prediction_dict = json.load(f)

train_df["family_prediction"] = train_df.apply(
    lambda x: family_to_date_prediction_dict[x["family"]][x["date"]], axis=1
)

XGBoost Classification Model

In [122]:
# Train test split
X = train_df.drop(columns=["id", "sales", "date"])
y = train_df["sales"]

split = int(0.75 * len(X))
X_train, X_val = X[:split], X[split:]
y_train, y_val = y[:split], y[split:]

In [123]:
X_train = X
y_train = y
X_val = X
y_val = y

In [124]:
model = XGBRegressor(
    n_estimators=800,
    learning_rate=0.03,
    max_depth=0,
    max_leaves=256,
    min_child_weight=1,
    gamma=0,
    subsample=0.8,
    colsample_bytree=0.8,
    grow_policy="lossguide",
    reg_lambda=0.1,
    enable_categorical=True,
    random_state=42,
)

model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=True, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=0, grow_policy='lossguide',
             importance_type=None, interaction_constraints=None,
             learning_rate=0.03, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=0,
             max_leaves=256, min_child_weight=1, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=800,
             n_jobs=None, num_parallel_tree=None, ...)

In [125]:

y_pred = model.predict(X_val)
y_pred = np.maximum(0, y_pred)

rmsle = np.sqrt(mean_squared_error(np.log1p(y_val), np.log1p(y_pred)))
print(f"Root Mean Squared Error (RMSE): {rmsle:.4f}")

Root Mean Squared Error (RMSE): 0.6933


Model Inference

In [126]:
test_df = pd.read_csv("data/test.csv")

In [128]:
test_df = process_train_data(test_df, has_holiday_dict, oil_prices_dict, stores_df, key_to_transactions)

In [129]:
# Conforming test_df schema

test_df["store_family_prediction"] = np.nan
test_df["store_prediction"] = np.nan
test_df["family_prediction"] = np.nan
test_df["sales"] = np.nan
test_df = test_df[train_df.columns]

In [ ]:
def model_inference(test_df, train_df): 
    """Model inference loop (Highly unoptimized, would be more optimized if models can be preloaded)"""
    
    dates = sorted(test_df["date"].unique().tolist(), reverse=True)

    sales_forecasts = []
    while dates:
        
        # Fetch the processed dataframes
        category1_dfs_dict, scale_factors1 = process_lstm_data(train_df, grouping_category=1)
        category2_dfs_dict, scale_factors2 = process_lstm_data(train_df, grouping_category=2)
        category3_dfs_dict, scale_factors3 = process_lstm_data(train_df, grouping_category=3)
        
        # Shorten the dataframes to the latest 20
        for key in category1_dfs_dict.keys():
            category1_dfs_dict[key] = category1_dfs_dict[key][-21:]
        for key in category2_dfs_dict.keys():
            category2_dfs_dict[key] = category2_dfs_dict[key][-21:]
        for key in category3_dfs_dict.keys():
            category3_dfs_dict[key] = category3_dfs_dict[key][-21:]   

        # Fetch current date
        curr_date = dates.pop()

        # Run LSTM Inference

        next_values1 = defaultdict(int)
        for key in category1_dfs_dict.keys():
            curr_df = category1_dfs_dict[key]
            X, y = preprocess_lstm_data(curr_df, 20)

            curr_model = tensorflow.keras.models.load_model(f"my_models/specialist_model_{key}", compile=False)
            X_inference = X[-20:]
            y_pred = curr_model.predict(X_inference, verbose=0)

            del curr_model

            y_pred = y_pred[0][0]
            y_pred *= scale_factors1[key]

            next_values1[key] = y_pred
        
        next_values2 = defaultdict(int)
        for key in category2_dfs_dict.keys():
            curr_df = category2_dfs_dict[key]
            X, y = preprocess_lstm_data(curr_df, 20)

            curr_model = tensorflow.keras.models.load_model(f"my_models/specialist_model_{key}", compile=False)
            X_inference = X[-20:]
            y_pred = curr_model.predict(X_inference, verbose=0)

            del curr_model

            y_pred = y_pred[0][0]
            y_pred *= scale_factors2[key]

            next_values2[key] = y_pred

        next_values3 = defaultdict(int)
        for key in category3_dfs_dict.keys():
            curr_df = category3_dfs_dict[key]
            X, y = preprocess_lstm_data(curr_df, 20)

            curr_model = tensorflow.keras.models.load_model(f"my_models/specialist_model_{key}", compile=False)
            X_inference = X[-20:]
            y_pred = curr_model.predict(X_inference, verbose=0)

            del curr_model

            y_pred = y_pred[0][0]
            y_pred *= scale_factors3[key]

            next_values3[key] = y_pred


        # Populate test dataframe at date = curr_date

        mask = test_df["date"] == curr_date

        test_df.loc[mask, "store_family_prediction"] = test_df.apply(
            lambda x: next_values1[(x["store_nbr"], x["family"])], axis=1
        )

        test_df.loc[mask, "store_prediction"] = test_df.apply(
            lambda x: next_values2[x["store_nbr"]], axis=1
        )

        test_df.loc[mask, "family_prediction"] = test_df.apply(
            lambda x: next_values3[x["family"]], axis=1
        )

        # XGBoost Inference
        inference_df = test_df[test_df["date"] == curr_date]
        inference_df_filtered = inference_df[X_train.columns]
        y_pred = model.predict(inference_df_filtered)
        
        # Save the predictions
        sales_forecasts.append(y_pred)
        inference_df["sales"] = y_pred

        # Updates train_df
        train_df = pd.concat([train_df, inference_df])

        print(y_pred)

    return sales_forecasts
    
sales_forecasts2 = model_inference(test_df, train_df)

In [131]:
sales_forecasts_unpacked = []

for arr in sales_forecasts2:
    sales_forecasts_unpacked.extend(arr)

In [132]:
test_df["sales"] = sales_forecasts_unpacked
test_df = test_df[["id", "sales"]]
test_df["sales"] = test_df["sales"].clip(lower=0)

In [133]:
test_df.to_csv("submission.csv", index=False)

Single LSTM Model Approach

In [16]:
# Fetching train data

train_df = pd.read_csv("data/train.csv")
train_df = train_df[train_df["sales"].notna()]

In [17]:
# Fetching supplamentary data

has_holiday_dict = processed_holidays_events()
oil_prices_dict = process_oil_prices()
stores_df = process_stores_table()
key_to_transactions = process_transactions()

In [18]:
train_df = process_train_data(train_df, has_holiday_dict, oil_prices_dict, stores_df, key_to_transactions)

In [19]:
# Converting family column to int

train_df["family"], uniques = pd.factorize(train_df["family"])
train_df["family"] += 1        

In [51]:
category1_dfs, scale_factors1 = process_lstm_data(train_df, grouping_category=1)
category2_dfs, scale_factors2 = process_lstm_data(train_df, grouping_category=2)
category3_dfs, scale_factors3 = process_lstm_data(train_df, grouping_category=3)

In [42]:
all_dfs = {}
all_dfs.update(category1_dfs)
all_dfs.update(category2_dfs)
all_dfs.update(category3_dfs)

all_scale_factors = {}
all_scale_factors.update(scale_factors1)
all_scale_factors.update(scale_factors2)
all_scale_factors.update(scale_factors3)

In [61]:
# Defining the model

seq_len = 20
num_features = 7
hidden_dim = 64

num_stores = 54       
num_products = 33  

store_emb_dim = 8
product_emb_dim = 8

# Embedding Inputs
store_input = layers.Input(shape=(seq_len,), dtype="int32", name="store_id")
product_input = layers.Input(shape=(seq_len,), dtype="int32", name="product_id")
seq_input = layers.Input(shape=(seq_len, num_features), name="sequence_data")

# Embeddings
store_emb = layers.Embedding(
    input_dim=num_stores + 1,
    output_dim=store_emb_dim
)(store_input)

product_emb = layers.Embedding(
    input_dim=num_products + 1,
    output_dim=product_emb_dim
)(product_input)
                         
# Combining features
x = layers.Concatenate(axis=-1)([
    seq_input,
    store_emb,
    product_emb
])

# Defining the LSTM 
x = layers.LSTM(hidden_dim, return_sequences=True, dropout=0.2)(x)
x = layers.LSTM(hidden_dim, return_sequences=True, dropout=0.2)(x)
x = layers.LSTM(hidden_dim, dropout=0.2)(x)

output = layers.Dense(1, activation="linear")(x)

model = models.Model(
    inputs=[store_input, product_input, seq_input],
    outputs=output
)

model.compile(
    optimizer=Adam(learning_rate=0.01),
    loss="mse",
    metrics=[tensorflow.keras.metrics.RootMeanSquaredError()]
)

In [54]:
X_all = []
y_all = []

store_ids = []
product_ids = []

seq_len = 20

for key, grouped_data in category1_dfs.items():
    X, y = preprocess_lstm_data(grouped_data, seq_len)

    X_all.append(X)
    y_all.append(y)

    n_seq = len(X)

    store_ids.append(np.full((n_seq, seq_len), key[0], dtype=np.int32))
    product_ids.append(np.full((n_seq, seq_len), key[1], dtype=np.int32))


X_all = np.concatenate(X_all, axis=0)
y_all = np.concatenate(y_all, axis=0)

store_ids = np.concatenate(store_ids, axis=0) 
product_ids = np.concatenate(product_ids, axis=0)


In [66]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor= "val_root_mean_squared_error",
    patience=3,     
    restore_best_weights=True
)

history = model.fit(
    x=[store_ids, product_ids, X_all],
    y=y_all,
    batch_size=32,
    epochs=5,
    validation_split=0.2,
    shuffle=True,
    callbacks = [early_stop]
)

Epoch 1/5
74132/74132 ━━━━━━━━━━━━━━━━━━━━ 1139s 15ms/step - loss: 0.0168 - root_mean_squared_error: 0.1295 - val_loss: 0.0353 - val_root_mean_squared_error: 0.1879
Epoch 2/5
74132/74132 ━━━━━━━━━━━━━━━━━━━━ 1147s 15ms/step - loss: 0.0337 - root_mean_squared_error: 0.1835 - val_loss: 0.0375 - val_root_mean_squared_error: 0.1935
Epoch 3/5
74132/74132 ━━━━━━━━━━━━━━━━━━━━ 1022s 14ms/step - loss: 0.0372 - root_mean_squared_error: 0.1928 - val_loss: 0.0394 - val_root_mean_squared_error: 0.1986
Epoch 4/5
74132/74132 ━━━━━━━━━━━━━━━━━━━━ 1037s 14ms/step - loss: 0.0380 - root_mean_squared_error: 0.1950 - val_loss: 0.0427 - val_root_mean_squared_error: 0.2066


In [68]:
model.save("singular_lstm_model.keras")

In [70]:
from tensorflow.keras.models import load_model

loaded_model = load_model("singular_lstm_model.keras")